In [1]:
!nvidia-smi

Thu Jan 22 19:38:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.14              Driver Version: 550.54.14      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:17:00.0 Off |                    0 |
|  0%   33C    P0             53W /  300W |      19MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys

sys.path.append("..")

In [3]:
import itertools
import time
from typing import Literal
from pathlib import Path

from vllm import LLM
from vllm.entrypoints.chat_utils import ChatCompletionMessageParam
from vllm.sampling_params import SamplingParams, StructuredOutputsParams

from src.audio_utils import get_audio_transcript
from src.config import InferenceConfig
from src.inference_utils import CATEGORIES, create_messages
from src.postprocessing_utils import postprocess_response


In [4]:
llm = LLM(
    model="HuggingFaceTB/SmolVLM2-2.2B-Instruct",
    dtype="bfloat16",
    gpu_memory_utilization=0.3,
    seed=42,
    enforce_eager=True,
    allowed_local_media_path=str(Path("../data/inputs").resolve()),
    limit_mm_per_prompt={"video": 10, "videos": 10},
)


INFO 01-22 19:40:38 [utils.py:263] non-default args: {'allowed_local_media_path': '/AIML/tinyllms/work/ETLLM/VLM-Inference/data/inputs', 'dtype': 'bfloat16', 'seed': 42, 'gpu_memory_utilization': 0.3, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'video': 10, 'videos': 10}, 'model': 'HuggingFaceTB/SmolVLM2-2.2B-Instruct'}
INFO 01-22 19:42:07 [model.py:530] Resolved architecture: SmolVLMForConditionalGeneration
INFO 01-22 19:42:07 [model.py:1866] Downcasting torch.float32 to torch.bfloat16.
INFO 01-22 19:42:07 [model.py:1545] Using max model len 8192
INFO 01-22 19:42:09 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 01-22 19:42:09 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 01-22 19:42:09 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
WARNING 01-22 19:42:09 [vllm.py:665] Enforce eager set, overriding optimization level to -O0
INFO 01-22 19:42:09 [vllm.py:765] Cudagraph is disab

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore_DP0 pid=3249332) INFO 01-22 19:42:37 [default_loader.py:291] Loading weights took 1.43 seconds
(EngineCore_DP0 pid=3249332) INFO 01-22 19:42:37 [gpu_model_runner.py:3905] Model loading took 4.2 GiB memory and 8.733939 seconds
(EngineCore_DP0 pid=3249332) INFO 01-22 19:42:38 [gpu_model_runner.py:4715] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 5 image items of the maximum feature size.
(EngineCore_DP0 pid=3249332) INFO 01-22 19:42:44 [gpu_worker.py:358] Available KV cache memory: 7.21 GiB
(EngineCore_DP0 pid=3249332) INFO 01-22 19:42:44 [kv_cache_utils.py:1305] GPU KV cache size: 39,344 tokens
(EngineCore_DP0 pid=3249332) INFO 01-22 19:42:44 [kv_cache_utils.py:1310] Maximum concurrency for 8,192 tokens per request: 4.80x
(EngineCore_DP0 pid=3249332) INFO 01-22 19:42:44 [core.py:273] init engine (profile, create kv cache, warmup model) took 6.55 seconds
(EngineCore_DP0 pid=3249332) WARNING 01-22 19:42:45 [vllm.py:672] Inductor compilatio

In [9]:
sampling_params = SamplingParams(
    max_tokens=140,
    temperature=0,
    seed=42,
)


In [10]:
video_folder = Path("../data/inputs/videos")

video_id_iter = video_folder.glob("*.mp4")
video_ids = tuple(video_id_iter)[:1]
print(f"Processing {len(video_ids)} videos...")

Processing 1 videos...


In [11]:
summary_messages: list[list[dict[str, str | list[dict[str, str]]]]] = []
category_messages: list[list[dict[str, str | list[dict[str, str]]]]] = []

audio_folder = Path("../data/inputs/audios")
audio_transcript_folder = Path("../data/inputs/audio_transcripts")

for video_path in video_ids:
    video_id = video_path.stem

    transcript = get_audio_transcript(
        video_id,
        video_path,
        audio_folder,
        audio_transcript_folder,
    )

    # ---- Summary Prompts ----
    summary_msg = create_messages(
        video_path, transcript, mode="summary", from_vllm=True
    )
    summary_messages.append(summary_msg)

    # ---- Category Prompts ----
    category_msg = create_messages(
        video_path, transcript, mode="category", from_vllm=True
    )
    category_messages.append(category_msg)

In [12]:
outputs = llm.chat(summary_messages, sampling_params)

ValueError: At most 0 video(s) may be provided in one prompt.